# RapidFire AI SFT Baseline for Schema Linking

This notebook adapts the RapidFire AI SFT tutorial to Project 2. The goal is a first end-to-end baseline: Qwen2.5-0.5B-Instruct, LoRA, raw JSON output, candidate-table schema serialization, and one epoch over the released training split.


In [19]:
import json
from datasets import Dataset
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from schema_linking_utils import make_messages


## Load Train and Validation Splits


In [20]:
def load_dataset_from_json(path):
    with open(path) as f:
        return Dataset.from_list(json.load(f))

train_dataset = load_dataset_from_json("train.json").shuffle(seed=42)
eval_dataset = load_dataset_from_json("validation.json").shuffle(seed=42)

len(train_dataset), len(eval_dataset)


(301, 101)

## Format Examples as Chat SFT Records

The prompt uses a compact candidate-schema dump without column types. The completion is exactly the schema-links JSON object. Tables and columns are sorted to make decoding targets stable. `None` values introduced by HuggingFace dataset struct expansion are skipped; real empty lists are preserved for table-only links.


In [21]:
def formatting_function(row):
    from schema_linking_utils import make_messages

    return make_messages(
        row,
        schemas_dir="schemas",
        include_completion=True,
        schema_mode="candidate",
        max_candidate_tables=30,
    )

formatting_function(train_dataset[0])


def supports_bf16():
    import torch
    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()


## Create RapidFire Experiment


In [22]:
experiment = Experiment(experiment_name="p2-qwen25-05b-lora-candidate30-json-1epoch", mode="fit")


The previously running experiment p2-qwen25-05b-lora-json-1epoch_4 was forcibly ended. Created a new experiment 'p2-qwen25-05b-lora-json-1epoch_5' with Experiment ID: 6 and Metric Experiment ID: 23 at /home/obychenkov/rapidfireai/rapidfire_experiments/p2-qwen25-05b-lora-json-1epoch_5


## Define Baseline LoRA and SFT Config


In [23]:
peft_configs = List([
    RFLoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
    )
])

config_set = List([
    RFModelConfig(
        model_name="Qwen/Qwen2.5-0.5B-Instruct",
        peft_config=peft_configs,
        training_args=RFSFTConfig(
            output_dir="rapidfire_outputs/qwen25_05b_lora_candidate30_json_1epoch",
            learning_rate=2e-4,
            lr_scheduler_type="linear",
            num_train_epochs=1,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            max_length=16384,
            logging_steps=5,
            eval_strategy="epoch",
            save_strategy="epoch",
            bf16=supports_bf16(),
            gradient_checkpointing=True,
            packing=False,
        ),
        model_type="causal_lm",
        model_kwargs={"use_cache": False},
        formatting_func=formatting_function,
    )
])


## Model Factory


In [24]:
def create_model(model_config):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = dict(model_config["model_kwargs"])
    if torch.cuda.is_available():
        model_kwargs.setdefault("torch_dtype", torch.bfloat16)
        model_kwargs.setdefault("device_map", "auto")

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return model, tokenizer


## Run SFT

This launches one RapidFire SFT config. After training, copy the best adapter into `adapter/` at the repo root so `main.py` can load it by default.


In [25]:
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")
experiment.run_fit(config_group, create_model, train_dataset, eval_dataset, num_chunks=1, seed=42)


Started 1 worker processes successfully
Created workers
INFO 05-29 18:31:30 [__init__.py:216] Automatically detected platform cuda.


## End Experiment


In [ ]:
experiment.end()
